# Day 38: Build your first autonomous Agent using LangGraph

Welcome to Day 38! Today we are building an **Autonomous Agent** using **LangGraph**. An agent is not just a standard LLM call; it is an LLM that has access to *tools* and can *reason* about when and how to use them.

## Core Theory (Just-in-Time)

### The "Why"
Standard LLMs have cut-off dates for their knowledge and struggle with precise mathematics or executing actions in the real world. By giving them "tools" (like a web search API or a calculator), we empower them to solve complex problems that require external data or exact computation.

### The "How"
We are using **LangGraph**, the modern, state-based way to build agentic loops. Older approaches (like `AgentExecutor` in early LangChain) were often "black boxes" that were hard to debug. LangGraph models the agent as a state machine:
1.  **State:** The memory of the agent (typically a list of messages).
2.  **Nodes:** Python functions that perform work (e.g., calling the LLM, executing a tool).
3.  **Edges:** The logic that connects nodes (e.g., "If the LLM says use a tool, go to the tool node; otherwise, go to the END node").

We will use:
-   `StateGraph` and `MessagesState` to hold the conversation.
-   `ChatOpenAI` (or another LLM) wrapped with `.bind_tools()`.
-   `ToolNode` and `tools_condition` from `langgraph.prebuilt` to handle tool execution and routing.

## Common Pitfalls in Production
1.  **Infinite Loops:** An agent might get stuck repeatedly calling a tool that fails. Always set a recursion limit when invoking LangGraph (`{"recursion_limit": 10}`).
2.  **Tool Descriptions:** The LLM decides to use a tool based entirely on its docstring/description. If the description is vague, the LLM will hallucinate arguments or call the wrong tool.
3.  **Security (Remote Code Execution):** If you give an agent a Python REPL or `eval()` tool, you must sandbox it! In our calculator example, we strictly whitelist characters to prevent malicious code execution.
4.  **Deprecated Frameworks:** Avoid `ToolExecutor` and `ToolInvocation` (from LangGraph 0.1.x) and `AgentExecutor` (from LangChain). Use `ToolNode` and `tools_condition`.


## 1. Setup and Dependencies

Let's install the necessary packages and set up our environment.

In [1]:
# uv pip install langchain langchain-openai langgraph duckduckgo-search ddgs

import os
import re
from typing import List, Annotated, Sequence

# We will use DuckDuckGo for search, which is free and requires no API key.
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

# Set a dummy key if not present (for local testing without a real OpenAI key)
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = "sk-dummy-key"


/tmp/ipykernel_31750/1183112939.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


## 2. Defining Tools

We will define two tools:
1.  A web search tool using DuckDuckGo.
2.  A secure calculator tool.

Notice how we use the `@tool` decorator. The **docstring** is critical—it tells the LLM exactly what the tool does.

In [2]:
# 1. Search Tool
# We use DuckDuckGo as it doesn't require an API key for demonstration.
search_tool = DuckDuckGoSearchRun(
    name="web_search",
    description="Useful for searching the web for current events, facts, or information not in your training data."
)

# 2. Calculator Tool
@tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression and returns the result.
    Use this for exact math calculations.
    Input must be a valid mathematical expression using numbers and basic operators (+, -, *, /, **).
    """
    # WARNING: PRODUCTION SAFEGUARD
    # We strictly whitelist characters to prevent remote code execution via eval()
    allowed_chars = set("0123456789+-*/(). ")
    
    if not all(char in allowed_chars for char in expression):
        return "Error: Invalid characters in math expression. Only numbers and basic operators are allowed."
    
    try:
        # We use eval securely because we pre-validated the input characters
        result = eval(expression, {"__builtins__": None}, {})
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {str(e)}"

# List of tools we will give to the agent
tools = [search_tool, calculator]


## 3. Building the LangGraph Agent

Now we construct the state machine.

1.  **State:** We use a simple `TypedDict` containing a list of messages. We use `Annotated[list, add_messages]` so that new messages append to the list rather than overwriting it.
2.  **LLM Node:** We bind the tools to the LLM and call it.
3.  **Tool Node:** LangGraph's prebuilt `ToolNode` handles executing the functions.
4.  **Routing:** `tools_condition` automatically routes to the ToolNode if the LLM requests a tool call, otherwise it routes to `END`.

In [3]:
class State(TypedDict):
    # The `add_messages` reducer ensures messages are appended to the list
    messages: Annotated[list[BaseMessage], add_messages]

def create_agent_graph(llm_tools: list):
    """Builds and returns the LangGraph application for the provided tools."""
    # 1. Initialize the LLM and bind the tools to it
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    llm_with_tools = llm.bind_tools(llm_tools)
    
    # 2. Define the LLM reasoning node
    def reasoner(state: State):
        """Calls the LLM to decide the next step."""
        try:
            response = llm_with_tools.invoke(state["messages"])
            return {"messages": [response]}
        except Exception as e:
            # Graceful fallback for local testing without valid API keys
            print(f"LLM Error: {e}")
            return {"messages": [AIMessage(content=f"LLM API Error: {str(e)}")]}

    # 3. Create the graph
    graph_builder = StateGraph(State)
    
    # 4. Add nodes
    graph_builder.add_node("reasoner", reasoner)
    
    # The ToolNode executes the tools based on the LLM's tool_calls
    tool_node = ToolNode(tools=llm_tools)
    graph_builder.add_node("tools", tool_node)
    
    # 5. Define edges
    # Start goes to the reasoner
    graph_builder.add_edge(START, "reasoner")
    
    # After reasoning, we use the prebuilt conditional edge `tools_condition`
    # If the LLM returned a tool_call, it goes to "tools". Otherwise, it goes to END.
    graph_builder.add_conditional_edges(
        "reasoner",
        tools_condition,
    )
    
    # After tool execution, always return to the reasoner to evaluate the result
    graph_builder.add_edge("tools", "reasoner")
    
    # Compile the graph into a runnable application
    return graph_builder.compile()

app = create_agent_graph(tools)

# Optional: You can visualize the graph if you have graphviz installed or use LangSmith.
# print(app.get_graph().draw_ascii())


## 4. Running the Agent

Let's test the agent with a complex query that requires both searching and calculating.

In [4]:
def run_agent(agent_app, query: str):
    """Helper function to run the graph and print the conversation."""
    print(f"\n--- User Query: {query} ---")
    
    # Initialize the state with the human message
    initial_state = {"messages": [HumanMessage(content=query)]}
    
    # Run the graph. We set a recursion limit to prevent infinite loops.
    try:
        # stream() yields state updates after every node execution
        for event in agent_app.stream(initial_state, {"recursion_limit": 10}):
            for node_name, state_update in event.items():
                print(f"\n[Node Execution: {node_name}]")
                # Print the latest message added by this node
                latest_msg = state_update["messages"][-1]
                
                if isinstance(latest_msg, AIMessage):
                    if latest_msg.tool_calls:
                        print(f"🤖 AI called tools: {latest_msg.tool_calls}")
                    else:
                        print(f"🤖 AI Answer: {latest_msg.content}")
                
                elif isinstance(latest_msg, ToolMessage):
                    print(f"🛠️ Tool Result ({latest_msg.name}): {latest_msg.content}")
                    
    except Exception as e:
        print(f"Execution failed: {e}")

# Note: This will likely fail gracefully if using a dummy API key, 
# but the structure is 100% production-ready.
run_agent(app, "What is the current population of the US multiplied by 2?")



--- User Query: What is the current population of the US multiplied by 2? ---


LLM Error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

[Node Execution: reasoner]
🤖 AI Answer: LLM API Error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}


## 5. Practical Lab / Homework

Your task is to add a new tool to the agent.

**Task:**
1.  Create a new `@tool` called `get_current_weather` that takes a `city: str` as input.
2.  Implement the tool using a mock dictionary. E.g., if the city is "New York", return "75°F and Sunny". If the city is "London", return "60°F and Rainy". Otherwise, return "Weather unknown".
3.  Add this new tool to the `tools` list.
4.  Re-run the agent with the query: `"What is the weather in London right now?"`


In [5]:
@tool
def get_current_weather(city: str) -> str:
    """
    Gets the current weather for a given city.
    """
    city_weather_data = {
        "new york": "75°F and Sunny",
        "london": "60°F and Rainy",
        "tokyo": "80°F and Clear",
        "paris": "65°F and Cloudy"
    }
    return city_weather_data.get(city.lower(), "Weather unknown for this location.")

# Add to tools list
all_tools = [search_tool, calculator, get_current_weather]

# Build a new application with the extended tools
app_with_weather = create_agent_graph(all_tools)

# Re-run the agent with the new tool available
run_agent(app_with_weather, "What is the weather in London right now?")



--- User Query: What is the weather in London right now? ---
LLM Error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

[Node Execution: reasoner]
🤖 AI Answer: LLM API Error: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
